# Day 4 — Loops, comprehensions, generators
Objectives:
- for/while loops.
- List/dict/set comprehensions.
- Generator functions and expressions.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-04`. Read
`python/ds-60day/companion-guides/day04_loops_comprehensions_generators.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

An iterable is a source that can provide values one at a time. A `for`
loop asks for each next value, binds it to a loop name, and runs the
indented body. Direct iteration communicates “use each value”; use
`enumerate` when both position and value matter, and `range` when you
truly need a sequence of integers.

A comprehension eagerly builds a new collection. Read
`[transform(item) for item in source if keep(item)]` from the middle:
take each item from the source, keep matching items, then transform
them. A generator uses `yield` or parentheses to produce values lazily.
It remembers its position, performs work only when consumed, and is
normally exhausted after one pass.

### Vocabulary

- **iterable:** an object able to provide an iterator, such as a list or range.
- **iterator:** a stateful one-way cursor that provides the next value.
- **iteration:** one pass through successive values.
- **comprehension:** compact syntax that eagerly constructs a collection.
- **generator:** an iterator that computes values lazily.
- **exhaustion:** the state after an iterator has no more values.

## Syntax anatomy

In `[n * n for n in numbers if n > 0]`, `n * n` is the output
expression, `for n in numbers` names the source and current item, and
`if n > 0` is the filter. The filter runs before the output expression.
In a generator function, each `yield value` pauses the function while
preserving local state; the next request resumes immediately after that
`yield`.

### Worked example 1 — Translate an append loop one clause at a time

Use the loop as a readable specification before compressing it. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
temperatures = [-4, 0, 7, 12]
warm_fahrenheit = []
for celsius in temperatures:
    if celsius > 0:
        warm_fahrenheit.append(celsius * 9 / 5 + 32)

compact = [c * 9 / 5 + 32 for c in temperatures if c > 0]
(warm_fahrenheit, compact, warm_fahrenheit == compact)

**Expected observation:** `([44.6, 53.6], [44.6, 53.6], True)`. The two forms implement the same filter and transformation.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Observe a generator's saved position

Consumption advances the iterator instead of restarting it. Predict first; then run the next cell.

In [ ]:
squares = (number**2 for number in range(4))
first = next(squares)
rest = list(squares)
after_exhaustion = list(squares)
(first, rest, after_exhaustion)

**Expected observation:** `(0, [1, 4, 9], [])`. Materializing `rest` consumes everything that remained.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Print or record the exact values produced by `range(start, stop, step)` when a boundary is wrong.
2. Expand a confusing comprehension back into loops and name its filter and transformation.
3. If a second pass is empty, check whether you retained an iterator instead of an iterable.
4. Never remove items from the same list being iterated; build a result or iterate over a copy.

**Alternative to compare:** Choose a normal loop for side effects or several decisions, a comprehension for one clear collection transformation, and a generator for streaming.

**Boundary to test:** Empty inputs, a non-positive batch size, a final partial batch, and inclusive versus exclusive endpoints must be explicit.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
squares = [i*i for i in range(10)]
squares

square_map = {i: i*i for i in range(5)}
square_map

unique_letters = {c for c in 'datascience'}
unique_letters


In [ ]:
def count_up_to(n):
    for i in range(1, n+1):
        yield i

gen = count_up_to(5)
list(gen)


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Start with `numbers = [-3, -1, 0, 2, 5]`. Write an ordinary loop that appends the **squares of strictly positive values** to `positive_squares`, then write an equivalent list comprehension named `compact_squares`.
   **Expected result:** both are `[4, 25]`. **Constraints:** do not mutate `numbers`, and keep the filter (`> 0`) distinct from the transformation (`value ** 2`).
   **Verify:** assert the two results are equal and the input is unchanged.

2. Implement `evens_through(limit)` as a generator that yields even integers from `0` through `limit` **inclusive**. **Inputs to verify:** `-1`, `0`, `1`, `2`, and `7`. **Expected results:** `[]`, `[0]`, `[0]`, `[0, 2]`, and `[0, 2, 4, 6]`. **Constraints:** use `yield`, return no stored list, and document the inclusive endpoint.
   **Verify:** Assert the exact five expected lists for limits `-1`, `0`, `1`, `2`, and `7`; also confirm `inspect.isgenerator(evens_through(2))` is true.

3. For `items = ['red', 'blue', 'red', 'green', 'blue', 'red']`, build a frequency mapping whose expected value is `{'red': 3, 'blue': 2, 'green': 1}`. **First:** implement an explicit one-pass loop with `dict.get`. **Then:** compare it with `collections.Counter`. **Constraint:** do not repeatedly call `items.count` in production code.
   **Verify:** assert both mappings agree and explain their time-cost difference.

### Additional mastery practice

Trace iteration boundaries and laziness. Prefer a clear loop when a comprehension would hide state changes or several decisions.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

4. **Prediction:** Create a generator expression, consume one item with `next`, then convert the rest to a list. Predict what remains and why.
   **Progressive hint:** Iterators remember their current position and are usually one-shot.
   **Verify:** Assert the first consumed square is `0`, the remaining list is `[1, 4, 9]`, and a third pass is empty; explain the saved iterator position.
5. **Tracing:** Trace `[n * 10 for n in range(6) if n % 2]` one input at a time.
   **Progressive hint:** Evaluate the filter before the output expression.
   **Verify:** Build a six-row trace for inputs `0..5` containing filter result and optional output; confirm the final list is `[10, 30, 50]`.
6. **Implementation:** Implement `batched(items, size)` yielding lists of at most `size`, including a final partial batch.
   **Progressive hint:** Accumulate, yield when full, then handle leftovers after the loop.
   **Verify:** Assert `list(batched(range(5), 2)) == [[0, 1], [2, 3], [4]]`, empty input yields no batches, and size `0` raises.
7. **Debugging:** Explain and repair a loop that removes negative values from the same list it is iterating over.
   **Progressive hint:** Build a new list or iterate over a copy.
   **Verify:** Keep the original failing list as evidence, then assert the repaired result removes every negative without skipping adjacent negatives or mutating the source unexpectedly.
8. **Edge case and explanation:** Define whether an even-number generator 'up to N' includes N and test N values -1, 0, 1, 2, and 3.
   **Progressive hint:** Boundary examples turn ambiguous English into a contract.
   **Verify:** Assert exact output for `N` values `-1, 0, 1, 2, 3`; identify which tests prove the endpoint is inclusive and how negative input behaves.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Start with `numbers = [-3, -1, 0, 2, 5]`. Write an ordinary loop that appends the **squares of strictly positive values** to `positive_squares`, then write an equivalent list comprehension named `compact_squares`. **Expected result:** both are `[4, 25]`. **Constraints:** do not mutate `numbers`, and keep the filter (`> 0`) distinct from the transformation (`value ** 2`). **Verify:** assert the two results are equal and the input is unchanged.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Start with `numbers = [-3, -1, 0, 2, 5]`. Write an ordinary loop that appends the **squares of strictly positive values** to `positive_squares`, then write an equivalent list co...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Implement `evens_through(limit)` as a generator that yields even integers from `0` through `limit` **inclusive**. **Inputs to verify:** `-1`, `0`, `1`, `2`, and `7`. **Expected results:** `[]`, `[0]`, `[0]`, `[0, 2]`, and `[0, 2, 4, 6]`. **Constraints:** use `yield`, return no stored list, and document the inclusive endpoint. **Verify:** Assert the exact five expected lists for limits `-1`, `0`, `1`, `2`, and `7`; also confirm `inspect.isgenerator(evens_through(2))` is true.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Implement `evens_through(limit)` as a generator that yields even integers from `0` through `limit` **inclusive**. `-1`, `0`, `1`, `2`, and `7`. `[]`, `[0]`, `[0]`, `[0, 2]`, and...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** For `items = ['red', 'blue', 'red', 'green', 'blue', 'red']`, build a frequency mapping whose expected value is `{'red': 3, 'blue': 2, 'green': 1}`. **First:** implement an explicit one-pass loop with `dict.get`. **Then:** compare it with `collections.Counter`. **Constraint:** do not repeatedly call `items.count` in production code. **Verify:** assert both mappings agree and explain their time-cost difference.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: For `items = ['red', 'blue', 'red', 'green', 'blue', 'red']`, build a frequency mapping whose expected value is `{'red': 3, 'blue': 2, 'green': 1}`. implement an explicit one-pa...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Create a generator expression, consume one item with `next`, then convert the rest to a list. Predict what remains and why. **Progressive hint:** Iterators remember their current position and are usually one-shot. **Verify:** Assert the first consumed square is `0`, the remaining list is `[1, 4, 9]`, and a third pass is empty; explain the saved iterator position.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Create a generator expression, consume one item with `next`, then convert the rest to a list. Predict what remains and why. Iterators remember their current position and are usu...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace `[n * 10 for n in range(6) if n % 2]` one input at a time. **Progressive hint:** Evaluate the filter before the output expression. **Verify:** Build a six-row trace for inputs `0..5` containing filter result and optional output; confirm the final list is `[10, 30, 50]`.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Trace `[n * 10 for n in range(6) if n % 2]` one input at a time. Evaluate the filter before the output expression. Build a six-row trace for inputs `0..5` containing filter resu...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement `batched(items, size)` yielding lists of at most `size`, including a final partial batch. **Progressive hint:** Accumulate, yield when full, then handle leftovers after the loop. **Verify:** Assert `list(batched(range(5), 2)) == [[0, 1], [2, 3], [4]]`, empty input yields no batches, and size `0` raises.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Implement `batched(items, size)` yielding lists of at most `size`, including a final partial batch. Accumulate, yield when full, then handle leftovers after the loop. Assert `li...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Explain and repair a loop that removes negative values from the same list it is iterating over. **Progressive hint:** Build a new list or iterate over a copy. **Verify:** Keep the original failing list as evidence, then assert the repaired result removes every negative without skipping adjacent negatives or mutating the source unexpectedly.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Explain and repair a loop that removes negative values from the same list it is iterating over. Build a new list or iterate over a copy. Keep the original failing list as eviden...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 8 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Define whether an even-number generator 'up to N' includes N and test N values -1, 0, 1, 2, and 3. **Progressive hint:** Boundary examples turn ambiguous English into a contract. **Verify:** Assert exact output for `N` values `-1, 0, 1, 2, 3`; identify which tests prove the endpoint is inclusive and how negative input behaves.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 8 — your work
# Short contract: Define whether an even-number generator 'up to N' includes N and test N values -1, 0, 1, 2, and 3. Boundary examples turn ambiguous English into a contract. Assert exact output...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
